In [ ]:
import sys
from pathlib import Path

# Go up until you reach the project root
PROJECT_ROOT = Path().resolve().parents[3]  # adjust number as needed
sys.path.append(str(PROJECT_ROOT))

from Tools.RunPlanning.memory_calculator import MemoryCalculator as MC

In [ ]:
import numpy as np

In [ ]:
# simulation dimension
sim_dim = 3

prob_lo = np.array([-5.0, -5.0]) * 1e-6  # simulation box: lower bounds
prob_hi = np.array([-5.0, 25.0]) * 1e-6  # simulation box: upper bounds

Nx, Ny, Nz = 2688, 1, 3712  # box size in cells

cell_size_z = (prob_hi[1] - prob_lo[1]) / Nz

# extent of the target in cells
target_x = Nx
target_y = Ny
target_z = np.ceil(2 * L_cut / cell_size_z).astype(int)

mc = MC(Nx, Ny, Nz, build_dim=sim_dim)

# memory allocated for electromagnetic and helper fields
field_mem = mc.mem_req_by_fields(
    Nx,
    Ny,
    Nz,
    divb_cleaning=True,
    dive_cleaning=True,
    pml_ncell=10,  # PML boundary conditions are active in this example
)

# particles per cell
species_e_ppc = 2 * 2 * 4
species_H_ppc = species_e_ppc
# memory allocated per particle species
species_e_mem = mc.mem_req_by_species(
    target_x,
    target_y,
    target_z,
    particles_per_cell=species_e_ppc,
)
species_H_mem = mc.mem_req_by_species(
    target_x,
    target_y,
    target_z,
    particles_per_cell=species_H_ppc,
    enable_ionization=True,  # H ions can be ionized
)
# memory allocated mainly for the states of RNGs
rng_mem = mc.mem_req_by_rng(warpx_compute="CUDA", gpu_model="A100")

# Print formatted summary with automatic field breakdown
total_mem = mc.print_summary(
    field_mem=field_mem,
    species_mems={"electrons": species_e_mem, "H ions": species_H_mem},
    rng_mem=rng_mem,
    show_breakdown=True,
)